# Part 2: Custom Inference & Attack Notebook

This notebook uses the custom `Model.load()` method to evaluate robustness.

In [ ]:
!pip install timm torchinfo matplotlib

In [ ]:
import torch
import torch.nn as nn
import timm
import os

class Model(nn.Module):
    def __init__(self, model_name='resnetv2_50x1_bit', num_classes=10):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=True)
        self.model.stem.conv = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.model.stem.pool = nn.Identity()
        for param in self.model.parameters():
            param.requires_grad = False
        for param in self.model.stem.conv.parameters():
            param.requires_grad = True
        for param in self.model.stages[3].parameters():
            param.requires_grad = True
        in_channels = self.model.head.fc.in_channels 
        self.model.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.4), nn.Linear(in_channels, num_classes)
        )
        for param in self.model.head.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        return self.model(x)

    @classmethod
    def load(cls, path, device='cpu', **kwargs):
        model = cls(**kwargs)
        model.load_state_dict(torch.load(path, map_location=device))
        model.to(device).eval()
        print(f"Model loaded from {path}")
        return model

In [ ]:
%%writefile attacks.py
import torch
import torch.nn.functional as F
class AdversarialAttack:
    def __init__(self, model, epsilon=0.1, alpha=0.01, iters=10):
        self.model = model; self.epsilon = epsilon; self.alpha = alpha; self.iters = iters
    def pgd(self, images, labels):
        images = images.clone().detach().to(images.device); labels = labels.clone().detach().to(images.device)
        ori_images = images.clone().detach()
        for i in range(self.iters):
            images.requires_grad = True; outputs = self.model(images); loss = F.cross_entropy(outputs, labels)
            self.model.zero_grad(); loss.backward(); adv_images = images + self.alpha * images.grad.data.sign()
            eta = torch.clamp(adv_images - ori_images, min=-self.epsilon, max=self.epsilon)
            images = (ori_images + eta).detach()
        return images

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from attacks import AdversarialAttack

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    model = Model.load("checkpoints/resnet_bit_cifar10.pth", device=DEVICE, num_classes=10)
except:
    print("Loading base model (no checkpoint found).")
    model = Model(num_classes=10).to(DEVICE)

transform_test = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
test_loader = DataLoader(datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test), batch_size=32, shuffle=True)

attacker = AdversarialAttack(model, epsilon=8/255.0)
data, target = next(iter(test_loader))
data, target = data.to(DEVICE), target.to(DEVICE)
data_adv = attacker.pgd(data, target)

with torch.no_grad():
    pred_clean = model(data).max(1)[1]
    pred_adv = model(data_adv).max(1)[1]

print(f"Clean Accuracy: {pred_clean.eq(target).sum().item()/32:.2%}")
print(f"Adversarial Accuracy: {pred_adv.eq(target).sum().item()/32:.2%}")